# Module 08: Advanced RAG Techniques

**Companion notebook for**: `08-advanced-rag.html`

## Learning Objectives
- Implement **query transformation** techniques: multi-query generation, HyDE, query decomposition
- Build **hybrid search** combining keyword (BM25) and semantic (vector) retrieval
- Apply **reranking** to improve retrieval precision after initial recall
- Construct **parent-child document retrieval** for balancing precision and context
- Generate **Hypothetical Document Embeddings (HyDE)** for improved retrieval
- Build a **self-corrective RAG** pipeline with relevance grading and query rewriting
- Implement **multi-query retrieval** with result fusion
- Use **contextual compression** to distill retrieved passages
- Evaluate retrieval quality with precision, recall, and MRR metrics
- Assemble an **end-to-end advanced RAG pipeline** combining multiple techniques

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Basic understanding of RAG concepts (Module 07)

---
## Setup & Dependencies

In [ ]:
%pip install -q openai chromadb rank-bm25 numpy tiktoken

In [ ]:
import os
import json
import uuid
import hashlib
from typing import Optional

import numpy as np
import chromadb
from openai import OpenAI
from rank_bm25 import BM25Okapi

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
client = OpenAI()

print("All imports successful. OpenAI API key found.")

---
## Sample Document Corpus

We create a realistic sample corpus of technical documents about AI/ML infrastructure.
This corpus will be used throughout the notebook to demonstrate each technique.

In [ ]:
# Sample documents simulating a technical knowledge base
DOCUMENTS = [
    {
        "id": "doc-1",
        "title": "PagedAttention and vLLM",
        "content": (
            "PagedAttention is a memory management technique for LLM inference that "
            "partitions the key-value cache into fixed-size pages stored in non-contiguous "
            "GPU memory. This eliminates memory fragmentation caused by variable-length "
            "sequences and enables near-zero waste of GPU memory. vLLM implements "
            "PagedAttention and achieves 2-4x higher throughput than HuggingFace "
            "Transformers for LLM serving. The system uses a block table to map logical "
            "KV cache blocks to physical GPU memory blocks, similar to virtual memory in "
            "operating systems."
        ),
        "category": "infrastructure"
    },
    {
        "id": "doc-2",
        "title": "RAG Chunking Strategies",
        "content": (
            "Effective chunking is critical for RAG system performance. Fixed-size chunking "
            "splits text at regular token intervals with overlap. Semantic chunking uses "
            "embedding similarity to find natural breakpoints. Recursive character splitting "
            "tries separators in order: paragraph breaks, newlines, sentences, then words. "
            "For technical documentation, 512-token chunks with 50-token overlap balance "
            "retrieval precision and context completeness. Larger chunks (1024-2048 tokens) "
            "provide more context but dilute retrieval signal."
        ),
        "category": "rag"
    },
    {
        "id": "doc-3",
        "title": "Vector Database Comparison",
        "content": (
            "ChromaDB is an open-source embedding database designed for AI applications. "
            "It supports in-memory and persistent storage modes. Pinecone is a managed "
            "vector database offering serverless deployment with automatic scaling. "
            "Weaviate provides hybrid search combining vector similarity with BM25 keyword "
            "search natively. Qdrant offers advanced filtering with payload indexing. For "
            "prototyping, ChromaDB is ideal. For production at scale, Pinecone or Weaviate "
            "handle billions of vectors with low latency."
        ),
        "category": "infrastructure"
    },
    {
        "id": "doc-4",
        "title": "Embedding Models Overview",
        "content": (
            "OpenAI text-embedding-3-small produces 1536-dimensional vectors and is the "
            "most cost-effective option for RAG applications. text-embedding-3-large offers "
            "higher quality at 3072 dimensions. Cohere embed-v3 supports 1024 dimensions "
            "with built-in compression. Open-source alternatives include BGE-large from "
            "BAAI and E5-mistral-7b-instruct which achieve competitive MTEB scores. "
            "Matryoshka representation learning allows truncating dimensions without "
            "retraining, enabling storage-performance tradeoffs."
        ),
        "category": "models"
    },
    {
        "id": "doc-5",
        "title": "Prompt Engineering for RAG",
        "content": (
            "RAG prompt engineering requires careful context placement and instruction "
            "design. Place retrieved context before the user question in the system or "
            "user message. Use XML tags or markdown headers to delimit context sections. "
            "Instruct the model to only answer from provided context and to say 'I don't "
            "know' when context is insufficient. Chain-of-thought prompting improves "
            "faithfulness by making the model cite specific passages. For multi-document "
            "QA, number the sources and ask the model to reference them."
        ),
        "category": "rag"
    },
    {
        "id": "doc-6",
        "title": "LLM Fine-Tuning with LoRA",
        "content": (
            "LoRA (Low-Rank Adaptation) inserts trainable rank decomposition matrices "
            "into transformer attention layers. Only the LoRA parameters are trained, "
            "reducing GPU memory by 60-80%% compared to full fine-tuning. QLoRA combines "
            "LoRA with 4-bit quantization using NormalFloat4 data type, enabling fine-tuning "
            "of 70B parameter models on a single 48GB GPU. Typical LoRA ranks of 16-64 "
            "achieve comparable quality to full fine-tuning for domain adaptation tasks."
        ),
        "category": "training"
    },
    {
        "id": "doc-7",
        "title": "Caching Strategies for LLM Applications",
        "content": (
            "Semantic caching stores LLM responses indexed by embedding similarity of the "
            "input prompt. When a new query is semantically similar to a cached query "
            "(cosine similarity > 0.95), the cached response is returned without an API "
            "call, reducing latency from seconds to milliseconds and cutting costs by "
            "40-70%%. Exact caching uses hash-based lookup for identical prompts. KV-cache "
            "reuse at the inference level shares prefix computations across requests. "
            "GPTCache and Redis with vector extensions are popular implementations."
        ),
        "category": "infrastructure"
    },
    {
        "id": "doc-8",
        "title": "Evaluation Metrics for RAG",
        "content": (
            "RAG evaluation requires metrics at both retrieval and generation stages. "
            "Retrieval metrics include precision@k (fraction of retrieved docs that are "
            "relevant), recall@k (fraction of relevant docs that are retrieved), and MRR "
            "(mean reciprocal rank of first relevant result). Generation metrics include "
            "faithfulness (are claims grounded in context?), relevance (does the answer "
            "address the question?), and completeness. RAGAS framework automates evaluation "
            "using LLM-as-judge. Human evaluation remains the gold standard for nuanced "
            "quality assessment."
        ),
        "category": "evaluation"
    },
    {
        "id": "doc-9",
        "title": "Guardrails for LLM Applications",
        "content": (
            "Guardrails protect LLM applications from generating harmful, biased, or "
            "off-topic content. Input guardrails filter malicious prompts using classifiers "
            "or regex patterns. Output guardrails validate response format, check for PII "
            "leakage, and verify factual consistency. NeMo Guardrails from NVIDIA provides "
            "programmable dialogue rails. Guardrails AI offers Pydantic-based output "
            "validation. Topic guardrails restrict the model to a defined scope using "
            "embedding similarity to an allowed topics list."
        ),
        "category": "safety"
    },
    {
        "id": "doc-10",
        "title": "Multi-Modal Document Processing",
        "content": (
            "Production documents contain tables, charts, diagrams, and images alongside "
            "text. Table extraction using pdfplumber or Camelot converts tables to markdown "
            "format preserving row-column structure. Vision models like GPT-4o describe "
            "charts and diagrams in text for embedding alongside document chunks. CLIP "
            "embeddings enable cross-modal retrieval where text queries match relevant "
            "images. ColPali uses vision-language models for end-to-end document retrieval "
            "without OCR. For best results, combine extracted text with image descriptions "
            "in a unified index."
        ),
        "category": "rag"
    }
]

print(f"Loaded {len(DOCUMENTS)} sample documents")
for doc in DOCUMENTS:
    print(f"  [{doc['id']}] {doc['title']} ({doc['category']})")

---
## Helper: Embedding Function & ChromaDB Setup

We set up a shared ChromaDB in-memory instance and a helper to generate embeddings via OpenAI.

In [ ]:
EMBED_MODEL = "text-embedding-3-small"


def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generate embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a_arr, b_arr = np.array(a), np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))


# Initialize ChromaDB in-memory client
chroma_client = chromadb.Client()

# Create the main collection and index all documents
main_collection = chroma_client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)

# Index documents
doc_texts = [doc["content"] for doc in DOCUMENTS]
doc_embeddings = get_embeddings(doc_texts)

main_collection.add(
    ids=[doc["id"] for doc in DOCUMENTS],
    embeddings=doc_embeddings,
    documents=doc_texts,
    metadatas=[{"title": doc["title"], "category": doc["category"]} for doc in DOCUMENTS]
)

print(f"Indexed {main_collection.count()} documents in ChromaDB")

---
## Section 1: Query Expansion / Transformation Techniques

Users rarely phrase questions in the exact way that matches document language.
Query transformation bridges this gap by rewriting, expanding, or decomposing queries
before they hit the vector store.

We implement three techniques:
1. **Query Rewriting** -- reformulates the question for better retrieval
2. **Multi-Query Generation** -- creates multiple phrasings and merges results
3. **Query Decomposition** -- breaks complex questions into simpler sub-questions

In [ ]:
def rewrite_query(question: str) -> str:
    """Rewrite a user question to be more effective for retrieval."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a search query optimizer. Rewrite the user question to be "
                    "more effective for document retrieval. Remove conversational fluff, "
                    "expand abbreviations, and add relevant technical terms. Return only "
                    "the rewritten query, nothing else."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.3,
        max_tokens=150,
    )
    return response.choices[0].message.content.strip()


def generate_multi_queries(question: str, n: int = 3) -> list[str]:
    """Generate N alternative phrasings of a question for retrieval."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    f"Generate {n} alternative versions of the given question for "
                    "document retrieval. Each version should use different keywords and "
                    "phrasing to maximize the chance of finding relevant documents. "
                    'Return as a JSON object with key "queries" containing a list of strings.'
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.7,
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result.get("queries", [question])


def decompose_query(question: str) -> list[str]:
    """Break a complex question into simpler sub-questions."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Break the complex question into 2-4 simpler sub-questions that "
                    "can be answered independently. Each sub-question should be self-contained. "
                    'Return as a JSON object with key "sub_questions" containing a list of strings.'
                ),
            },
            {"role": "user", "content": question},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result.get("sub_questions", [question])


# --- Demo ---
test_question = "Why is my RAG system returning irrelevant results?"

print("Original question:")
print(f"  {test_question}\n")

rewritten = rewrite_query(test_question)
print("Rewritten query:")
print(f"  {rewritten}\n")

multi_queries = generate_multi_queries(test_question)
print("Multi-query variants:")
for i, q in enumerate(multi_queries, 1):
    print(f"  {i}. {q}")

complex_question = (
    "How does our embedding model choice affect chunking strategy "
    "and what are the best practices for evaluating the combined impact?"
)
print(f"\nComplex question:\n  {complex_question}\n")

sub_questions = decompose_query(complex_question)
print("Decomposed sub-questions:")
for i, sq in enumerate(sub_questions, 1):
    print(f"  {i}. {sq}")

---
## Section 2: Hybrid Search (Keyword + Semantic)

Pure vector search can miss documents that share exact keywords but differ semantically,
while pure keyword search misses semantically similar content with different wording.
Hybrid search combines **BM25** (keyword) with **vector similarity** (semantic) and
merges results using **Reciprocal Rank Fusion (RRF)**.

In [ ]:
class HybridSearcher:
    """Combines BM25 keyword search with ChromaDB vector search."""

    def __init__(self, documents: list[dict], collection: chromadb.Collection):
        self.documents = documents
        self.collection = collection

        # Build BM25 index from document content
        tokenized_corpus = [doc["content"].lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)
        self.doc_ids = [doc["id"] for doc in documents]

    def keyword_search(self, query: str, k: int = 5) -> list[dict]:
        """BM25 keyword search."""
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)

        # Rank by score
        ranked_indices = np.argsort(scores)[::-1][:k]
        results = []
        for idx in ranked_indices:
            if scores[idx] > 0:
                results.append({
                    "id": self.doc_ids[idx],
                    "score": float(scores[idx]),
                    "content": self.documents[idx]["content"],
                    "title": self.documents[idx]["title"],
                })
        return results

    def semantic_search(self, query: str, k: int = 5) -> list[dict]:
        """ChromaDB vector similarity search."""
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "metadatas", "distances"],
        )
        output = []
        for i in range(len(results["ids"][0])):
            output.append({
                "id": results["ids"][0][i],
                "score": 1.0 - results["distances"][0][i],  # cosine distance -> similarity
                "content": results["documents"][0][i],
                "title": results["metadatas"][0][i]["title"],
            })
        return output

    def hybrid_search(
        self, query: str, k: int = 5, alpha: float = 0.5, rrf_k: int = 60
    ) -> list[dict]:
        """
        Hybrid search using Reciprocal Rank Fusion (RRF).

        Args:
            query: Search query
            k: Number of results to return
            alpha: Weight for semantic search (1-alpha for keyword)
            rrf_k: RRF constant (default 60)
        """
        keyword_results = self.keyword_search(query, k=k * 2)
        semantic_results = self.semantic_search(query, k=k * 2)

        # Build RRF scores
        rrf_scores: dict[str, float] = {}
        doc_map: dict[str, dict] = {}

        for rank, result in enumerate(keyword_results):
            doc_id = result["id"]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 - alpha) / (rrf_k + rank + 1)
            doc_map[doc_id] = result

        for rank, result in enumerate(semantic_results):
            doc_id = result["id"]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + alpha / (rrf_k + rank + 1)
            doc_map[doc_id] = result

        # Sort by combined RRF score
        sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:k]

        results = []
        for doc_id in sorted_ids:
            entry = doc_map[doc_id].copy()
            entry["rrf_score"] = rrf_scores[doc_id]
            results.append(entry)
        return results


# Initialize
hybrid_searcher = HybridSearcher(DOCUMENTS, main_collection)

# Compare results for a query where keyword and semantic differ
query = "vLLM KV cache memory management"

print(f"Query: '{query}'\n")
print("--- Keyword Search (BM25) ---")
for r in hybrid_searcher.keyword_search(query, k=3):
    print(f"  [{r['id']}] {r['title']} (score: {r['score']:.3f})")

print("\n--- Semantic Search (Vector) ---")
for r in hybrid_searcher.semantic_search(query, k=3):
    print(f"  [{r['id']}] {r['title']} (score: {r['score']:.3f})")

print("\n--- Hybrid Search (RRF, alpha=0.5) ---")
for r in hybrid_searcher.hybrid_search(query, k=3):
    print(f"  [{r['id']}] {r['title']} (rrf: {r['rrf_score']:.5f})")

---
## Section 3: Reranking Retrieved Results

Initial retrieval (whether keyword, semantic, or hybrid) optimizes for **recall** --
casting a wide net. Reranking then optimizes for **precision** by using a more
sophisticated model to re-score the retrieved candidates.

We implement an **LLM-based reranker** that scores each document's relevance to the query.
In production, dedicated cross-encoder models (Cohere Rerank, BGE-reranker) are faster.

In [ ]:
def rerank_with_llm(
    query: str, documents: list[dict], top_k: int = 3
) -> list[dict]:
    """
    Rerank documents using an LLM to score relevance.

    Each document is scored on a 0-10 scale for relevance to the query.
    Returns the top_k documents sorted by relevance score.
    """
    scored_docs = []

    for doc in documents:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Rate how relevant the following document is to the query. "
                        "Return a JSON object with a single key 'score' containing "
                        "an integer from 0 (not relevant) to 10 (perfectly relevant)."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Query: {query}\n\nDocument: {doc['content'][:500]}",
                },
            ],
            temperature=0.0,
            max_tokens=20,
            response_format={"type": "json_object"},
        )
        try:
            score = json.loads(response.choices[0].message.content)["score"]
        except (json.JSONDecodeError, KeyError):
            score = 0

        scored_docs.append({**doc, "relevance_score": score})

    # Sort by relevance score descending
    scored_docs.sort(key=lambda x: x["relevance_score"], reverse=True)
    return scored_docs[:top_k]


# Demo: Retrieve with hybrid, then rerank
query = "How to optimize GPU memory for serving large language models?"

print(f"Query: '{query}'\n")

# Step 1: Broad retrieval (5 candidates)
candidates = hybrid_searcher.hybrid_search(query, k=5)
print("Before reranking (hybrid retrieval):")
for i, doc in enumerate(candidates, 1):
    print(f"  {i}. [{doc['id']}] {doc['title']}")

# Step 2: Rerank to top 3
reranked = rerank_with_llm(query, candidates, top_k=3)
print("\nAfter reranking:")
for i, doc in enumerate(reranked, 1):
    print(f"  {i}. [{doc['id']}] {doc['title']} (relevance: {doc['relevance_score']}/10)")

---
## Section 4: Parent-Child Document Retrieval

The fundamental tension in chunking: **small chunks** retrieve precisely but lack context;
**large chunks** provide context but dilute retrieval signal.

Parent-child chunking resolves this by using small *child* chunks for retrieval (high
precision) but returning the *parent* chunk (rich context) to the LLM.

In [ ]:
class ParentChildRetriever:
    """
    Two-level retrieval: search small child chunks, return large parent chunks.

    - Parent chunks: large, context-rich (full document sections)
    - Child chunks: small, precise (for embedding & retrieval)
    """

    def __init__(self, parent_chunk_size: int = 500, child_chunk_size: int = 150):
        self.parent_chunk_size = parent_chunk_size
        self.child_chunk_size = child_chunk_size
        self.parent_store: dict[str, str] = {}  # parent_id -> parent_text
        self.parent_meta: dict[str, dict] = {}  # parent_id -> metadata

        # Create a dedicated child collection
        self.child_collection = chroma_client.get_or_create_collection(
            name="children",
            metadata={"hnsw:space": "cosine"},
        )

    def _split_text(self, text: str, chunk_size: int) -> list[str]:
        """Simple word-boundary chunking."""
        words = text.split()
        chunks = []
        current: list[str] = []
        current_len = 0
        for word in words:
            if current_len + len(word) + 1 > chunk_size and current:
                chunks.append(" ".join(current))
                current = []
                current_len = 0
            current.append(word)
            current_len += len(word) + 1
        if current:
            chunks.append(" ".join(current))
        return chunks

    def index_documents(self, documents: list[dict]):
        """Index documents with parent-child hierarchy."""
        all_child_ids = []
        all_child_texts = []
        all_child_metas = []

        for doc in documents:
            # Each document is a parent chunk
            parent_id = f"parent_{doc['id']}"
            self.parent_store[parent_id] = doc["content"]
            self.parent_meta[parent_id] = {
                "title": doc["title"],
                "category": doc["category"],
            }

            # Split into child chunks
            children = self._split_text(doc["content"], self.child_chunk_size)
            for i, child_text in enumerate(children):
                child_id = f"{parent_id}_c{i}"
                all_child_ids.append(child_id)
                all_child_texts.append(child_text)
                all_child_metas.append({
                    "parent_id": parent_id,
                    "title": doc["title"],
                })

        # Batch embed and add children
        child_embeddings = get_embeddings(all_child_texts)
        self.child_collection.add(
            ids=all_child_ids,
            embeddings=child_embeddings,
            documents=all_child_texts,
            metadatas=all_child_metas,
        )
        print(f"Indexed {len(documents)} parents -> {len(all_child_ids)} children")

    def retrieve(self, query: str, k: int = 3) -> list[dict]:
        """Search child chunks, return deduplicated parent chunks."""
        results = self.child_collection.query(
            query_texts=[query],
            n_results=k * 2,  # Over-fetch to account for deduplication
            include=["metadatas", "documents", "distances"],
        )

        seen_parents: set[str] = set()
        parents: list[dict] = []

        for i in range(len(results["ids"][0])):
            parent_id = results["metadatas"][0][i]["parent_id"]
            if parent_id not in seen_parents:
                seen_parents.add(parent_id)
                parents.append({
                    "parent_id": parent_id,
                    "parent_content": self.parent_store[parent_id],
                    "matched_child": results["documents"][0][i],
                    "child_distance": results["distances"][0][i],
                    "title": self.parent_meta[parent_id]["title"],
                })
            if len(parents) >= k:
                break

        return parents


# Build and demo
pc_retriever = ParentChildRetriever(parent_chunk_size=500, child_chunk_size=150)
pc_retriever.index_documents(DOCUMENTS)

query = "How does PagedAttention manage GPU memory?"
print(f"\nQuery: '{query}'\n")

results = pc_retriever.retrieve(query, k=2)
for r in results:
    print(f"Title: {r['title']}")
    print(f"Matched child ({len(r['matched_child'])} chars): {r['matched_child'][:100]}...")
    print(f"Returned parent ({len(r['parent_content'])} chars): {r['parent_content'][:100]}...")
    print()

---
## Section 5: Hypothetical Document Embeddings (HyDE)

HyDE asks the LLM to generate a *hypothetical answer* to the question, then embeds
that answer for retrieval instead of the question itself. The insight: a hypothetical
answer is semantically closer to actual documents than the question is, because
documents contain statements, not questions.

In [ ]:
def generate_hypothetical_document(question: str) -> str:
    """Generate a hypothetical document that would answer the question."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Write a short paragraph (3-5 sentences) that would appear in a "
                    "technical document answering the given question. Write it as factual "
                    "content, not as a response to a question. This will be used for "
                    "document retrieval."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.5,
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()


def hyde_search(
    question: str, collection: chromadb.Collection, k: int = 3
) -> tuple[str, list[dict]]:
    """
    HyDE retrieval: generate hypothetical doc, embed it, search.

    Returns (hypothetical_doc, search_results).
    """
    # Step 1: Generate hypothetical document
    hyde_doc = generate_hypothetical_document(question)

    # Step 2: Embed the hypothetical doc (not the question)
    hyde_embedding = get_embeddings([hyde_doc])[0]

    # Step 3: Search using the hypothetical doc embedding
    results = collection.query(
        query_embeddings=[hyde_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    output = []
    for i in range(len(results["ids"][0])):
        output.append({
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "distance": results["distances"][0][i],
        })

    return hyde_doc, output


# Demo: Compare standard vs HyDE retrieval
question = "How does PagedAttention work?"
print(f"Question: '{question}'\n")

# Standard retrieval
print("--- Standard Retrieval ---")
standard_results = main_collection.query(
    query_texts=[question], n_results=3, include=["metadatas", "distances"]
)
for i in range(len(standard_results["ids"][0])):
    title = standard_results["metadatas"][0][i]["title"]
    dist = standard_results["distances"][0][i]
    print(f"  {i+1}. {title} (distance: {dist:.4f})")

# HyDE retrieval
print("\n--- HyDE Retrieval ---")
hyde_doc, hyde_results = hyde_search(question, main_collection, k=3)
print(f"Hypothetical doc: {hyde_doc[:150]}...\n")
for i, r in enumerate(hyde_results, 1):
    print(f"  {i}. {r['title']} (distance: {r['distance']:.4f})")

---
## Section 6: Self-RAG with Relevance Checking

Standard RAG is one-shot: retrieve, generate, done. If retrieval returns irrelevant
documents, the answer suffers. **Self-corrective RAG** adds feedback loops:

1. **Retrieve** documents
2. **Grade** each document for relevance
3. If insufficient relevant docs, **rewrite** the query and retry
4. **Generate** answer from relevant context
5. **Check** for hallucinations against the context

This implements the CRAG (Corrective RAG) pattern.

In [ ]:
class SelfCorrectiveRAG:
    """Self-corrective RAG pipeline with relevance grading and query rewriting."""

    def __init__(self, collection: chromadb.Collection, max_retries: int = 2):
        self.collection = collection
        self.max_retries = max_retries

    def _retrieve(self, query: str, k: int = 5) -> list[dict]:
        """Retrieve documents from vector store."""
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "metadatas", "distances"],
        )
        return [
            {
                "id": results["ids"][0][i],
                "content": results["documents"][0][i],
                "title": results["metadatas"][0][i]["title"],
                "distance": results["distances"][0][i],
            }
            for i in range(len(results["ids"][0]))
        ]

    def _grade_relevance(self, query: str, document: str) -> bool:
        """Grade whether a document is relevant to the query."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Is this document relevant to the question?\n\n"
                        f"Question: {query}\n\n"
                        f"Document: {document[:500]}\n\n"
                        f'Answer with just "yes" or "no".'
                    ),
                }
            ],
            temperature=0.0,
            max_tokens=3,
        )
        return "yes" in response.choices[0].message.content.lower()

    def _rewrite_query(self, original_query: str) -> str:
        """Rewrite query for better retrieval after a failed attempt."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"The following question did not retrieve good results. "
                        f"Rewrite it to be more specific and use different keywords.\n\n"
                        f"Original: {original_query}\n\nRewritten:"
                    ),
                }
            ],
            temperature=0.7,
        )
        return response.choices[0].message.content.strip()

    def _check_hallucination(self, answer: str, context: str) -> bool:
        """Check if the answer is grounded in the provided context."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Is the following answer fully supported by the context? "
                        f"Answer 'grounded' if all claims are supported, or "
                        f"'hallucination' if any claims are not in the context.\n\n"
                        f"Context: {context[:1000]}\n\n"
                        f"Answer: {answer}"
                    ),
                }
            ],
            temperature=0.0,
            max_tokens=10,
        )
        return "grounded" in response.choices[0].message.content.lower()

    def _generate(self, query: str, context: str) -> str:
        """Generate answer from context."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Answer the question based only on the provided context. "
                        "If the context does not contain enough information, say so."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Context:\n{context}\n\nQuestion: {query}",
                },
            ],
            temperature=0.1,
        )
        return response.choices[0].message.content

    def query(self, question: str) -> dict:
        """
        Run the self-corrective RAG pipeline.

        Returns a dict with the answer, grading info, and pipeline trace.
        """
        trace: list[str] = []
        current_query = question

        for attempt in range(self.max_retries + 1):
            trace.append(f"Attempt {attempt + 1}: query = '{current_query}'")

            # Step 1: Retrieve
            docs = self._retrieve(current_query, k=5)
            trace.append(f"  Retrieved {len(docs)} documents")

            # Step 2: Grade relevance
            relevant_docs = []
            for doc in docs:
                is_relevant = self._grade_relevance(question, doc["content"])
                if is_relevant:
                    relevant_docs.append(doc)
                trace.append(
                    f"  [{doc['title']}] -> {'RELEVANT' if is_relevant else 'NOT RELEVANT'}"
                )

            # Step 3: Check if enough relevant docs
            if len(relevant_docs) >= 2 or attempt == self.max_retries:
                break

            # Step 4: Rewrite and retry
            current_query = self._rewrite_query(current_query)
            trace.append(f"  Insufficient relevant docs -> rewriting query")

        # Step 5: Generate answer
        context = "\n\n".join(doc["content"] for doc in relevant_docs)
        if not context:
            context = "\n\n".join(doc["content"] for doc in docs[:2])

        answer = self._generate(question, context)
        trace.append(f"  Generated answer ({len(answer)} chars)")

        # Step 6: Hallucination check
        is_grounded = self._check_hallucination(answer, context)
        trace.append(f"  Hallucination check: {'GROUNDED' if is_grounded else 'HALLUCINATION DETECTED'}")

        return {
            "question": question,
            "answer": answer,
            "is_grounded": is_grounded,
            "relevant_docs": [d["title"] for d in relevant_docs],
            "attempts": attempt + 1,
            "trace": trace,
        }


# Demo
self_rag = SelfCorrectiveRAG(main_collection, max_retries=2)

result = self_rag.query("How does PagedAttention improve LLM serving throughput?")

print("=" * 60)
print("Self-Corrective RAG Result")
print("=" * 60)
print(f"\nQuestion: {result['question']}")
print(f"\nAnswer: {result['answer']}")
print(f"\nGrounded: {result['is_grounded']}")
print(f"Attempts: {result['attempts']}")
print(f"Relevant docs: {result['relevant_docs']}")
print(f"\nPipeline trace:")
for step in result["trace"]:
    print(f"  {step}")

---
## Section 7: Multi-Query Retrieval

Multi-query retrieval generates multiple phrasings of the user's question, runs
independent retrievals for each, and merges the results. Different phrasings match
different documents, increasing recall without sacrificing precision.

In [ ]:
def multi_query_retrieve(
    question: str,
    collection: chromadb.Collection,
    n_queries: int = 3,
    k_per_query: int = 3,
    final_k: int = 5,
) -> list[dict]:
    """
    Multi-query retrieval with reciprocal rank fusion.

    1. Generate N alternative queries
    2. Run retrieval for each query
    3. Fuse results using RRF
    """
    # Step 1: Generate alternative queries
    queries = [question] + generate_multi_queries(question, n=n_queries)
    print(f"Queries ({len(queries)} total):")
    for i, q in enumerate(queries):
        label = "Original" if i == 0 else f"Generated {i}"
        print(f"  [{label}] {q}")
    print()

    # Step 2: Retrieve for each query
    all_results: dict[str, dict] = {}  # doc_id -> doc info
    rrf_scores: dict[str, float] = {}
    rrf_k = 60

    for q_idx, query in enumerate(queries):
        results = collection.query(
            query_texts=[query],
            n_results=k_per_query,
            include=["documents", "metadatas", "distances"],
        )
        for rank in range(len(results["ids"][0])):
            doc_id = results["ids"][0][rank]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (rrf_k + rank + 1)
            if doc_id not in all_results:
                all_results[doc_id] = {
                    "id": doc_id,
                    "title": results["metadatas"][0][rank]["title"],
                    "content": results["documents"][0][rank],
                }

    # Step 3: Sort by fused score
    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:final_k]
    return [
        {**all_results[doc_id], "rrf_score": rrf_scores[doc_id]}
        for doc_id in sorted_ids
    ]


# Demo
question = "What are the tradeoffs between different embedding models for RAG?"
print(f"Question: '{question}'\n")

results = multi_query_retrieve(question, main_collection)
print("Fused results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['id']}] {r['title']} (rrf: {r['rrf_score']:.5f})")

---
## Section 8: Contextual Compression

Retrieved chunks often contain irrelevant content alongside the useful information.
Contextual compression uses an LLM to extract only the portions of each retrieved
document that are relevant to the query, reducing noise in the generation context.

In [ ]:
def compress_document(query: str, document: str) -> Optional[str]:
    """
    Extract only the relevant parts of a document for the given query.

    Returns compressed text, or None if the document has no relevant content.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract only the sentences from the document that are directly "
                    "relevant to answering the question. If no part is relevant, "
                    "respond with 'NOT_RELEVANT'. Do not add any new information."
                ),
            },
            {
                "role": "user",
                "content": f"Question: {query}\n\nDocument: {document}",
            },
        ],
        temperature=0.0,
        max_tokens=300,
    )
    result = response.choices[0].message.content.strip()
    return None if result == "NOT_RELEVANT" else result


def retrieve_and_compress(
    query: str, collection: chromadb.Collection, k: int = 5
) -> list[dict]:
    """Retrieve documents and compress them to only relevant content."""
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas"],
    )

    compressed = []
    for i in range(len(results["ids"][0])):
        original = results["documents"][0][i]
        title = results["metadatas"][0][i]["title"]
        extracted = compress_document(query, original)

        if extracted:
            compressed.append({
                "id": results["ids"][0][i],
                "title": title,
                "original_length": len(original),
                "compressed_length": len(extracted),
                "compressed_content": extracted,
                "compression_ratio": len(extracted) / len(original),
            })

    return compressed


# Demo
query = "What embedding models are available and what are their dimensions?"
print(f"Query: '{query}'\n")

compressed_results = retrieve_and_compress(query, main_collection, k=3)
for r in compressed_results:
    print(f"[{r['title']}]")
    print(f"  Original: {r['original_length']} chars -> Compressed: {r['compressed_length']} chars")
    print(f"  Compression ratio: {r['compression_ratio']:.1%}")
    print(f"  Content: {r['compressed_content'][:200]}...")
    print()

---
## Section 9: Evaluation of Retrieval Quality

Systematic evaluation is critical for understanding whether your retrieval pipeline
is working. We implement three standard retrieval metrics:

- **Precision@k**: What fraction of the top-k retrieved docs are relevant?
- **Recall@k**: What fraction of all relevant docs appear in the top-k?
- **MRR (Mean Reciprocal Rank)**: How high is the first relevant result ranked?

In [ ]:
# Define evaluation dataset: queries with known relevant document IDs
EVAL_DATASET = [
    {
        "query": "How does vLLM manage GPU memory for LLM inference?",
        "relevant_ids": ["doc-1"],
    },
    {
        "query": "What chunking strategies work best for RAG?",
        "relevant_ids": ["doc-2", "doc-5"],
    },
    {
        "query": "Compare vector databases for production use",
        "relevant_ids": ["doc-3"],
    },
    {
        "query": "What embedding models should I use and what dimensions do they output?",
        "relevant_ids": ["doc-4"],
    },
    {
        "query": "How to evaluate RAG system performance with metrics?",
        "relevant_ids": ["doc-8"],
    },
    {
        "query": "How to fine-tune LLMs efficiently with limited GPU memory?",
        "relevant_ids": ["doc-6"],
    },
    {
        "query": "How to cache LLM responses to reduce cost and latency?",
        "relevant_ids": ["doc-7"],
    },
    {
        "query": "How to process tables and images in documents for RAG?",
        "relevant_ids": ["doc-10"],
    },
]


def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    """Fraction of top-k retrieved docs that are relevant."""
    top_k = retrieved_ids[:k]
    relevant_in_top_k = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return relevant_in_top_k / k


def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    """Fraction of relevant docs that appear in top-k."""
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    relevant_found = sum(1 for doc_id in relevant_ids if doc_id in top_k)
    return relevant_found / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """Reciprocal of the rank of the first relevant result."""
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def evaluate_retriever(
    search_fn,
    eval_dataset: list[dict],
    k: int = 3,
    label: str = "Retriever",
) -> dict:
    """Evaluate a retrieval function on the eval dataset."""
    precisions, recalls, rrs = [], [], []

    for item in eval_dataset:
        retrieved = search_fn(item["query"], k=k)
        retrieved_ids = [r["id"] for r in retrieved]

        precisions.append(precision_at_k(retrieved_ids, item["relevant_ids"], k))
        recalls.append(recall_at_k(retrieved_ids, item["relevant_ids"], k))
        rrs.append(reciprocal_rank(retrieved_ids, item["relevant_ids"]))

    metrics = {
        "label": label,
        f"precision@{k}": np.mean(precisions),
        f"recall@{k}": np.mean(recalls),
        "MRR": np.mean(rrs),
    }
    return metrics


# Define search functions to evaluate
def semantic_search_fn(query: str, k: int = 3) -> list[dict]:
    results = main_collection.query(
        query_texts=[query], n_results=k, include=["metadatas"]
    )
    return [{"id": results["ids"][0][i]} for i in range(len(results["ids"][0]))]


def hybrid_search_fn(query: str, k: int = 3) -> list[dict]:
    return hybrid_searcher.hybrid_search(query, k=k)


def keyword_search_fn(query: str, k: int = 3) -> list[dict]:
    return hybrid_searcher.keyword_search(query, k=k)


# Evaluate all retrievers
k = 3
print(f"Retrieval Evaluation (k={k})")
print("=" * 55)
print(f"{'Method':<20} {'Precision@k':<14} {'Recall@k':<12} {'MRR':<8}")
print("-" * 55)

for label, fn in [
    ("Keyword (BM25)", keyword_search_fn),
    ("Semantic (Vector)", semantic_search_fn),
    ("Hybrid (RRF)", hybrid_search_fn),
]:
    metrics = evaluate_retriever(fn, EVAL_DATASET, k=k, label=label)
    print(
        f"{label:<20} "
        f"{metrics[f'precision@{k}']:<14.3f} "
        f"{metrics[f'recall@{k}']:<12.3f} "
        f"{metrics['MRR']:<8.3f}"
    )

---
## Section 10: End-to-End Advanced RAG Pipeline

Now we combine multiple techniques into a single, configurable pipeline:

1. **Query transformation** (multi-query + HyDE)
2. **Hybrid retrieval** (BM25 + vector)
3. **Reranking** (LLM-based)
4. **Contextual compression**
5. **Self-corrective generation** with hallucination checking

This represents a production-grade RAG pipeline with the key advanced techniques.

In [ ]:
class AdvancedRAGPipeline:
    """
    End-to-end advanced RAG pipeline combining:
    - Multi-query generation
    - Hybrid search (BM25 + vector)
    - LLM reranking
    - Contextual compression
    - Hallucination checking
    """

    def __init__(
        self,
        hybrid_searcher: HybridSearcher,
        use_multi_query: bool = True,
        use_rerank: bool = True,
        use_compression: bool = True,
        use_hallucination_check: bool = True,
    ):
        self.hybrid_searcher = hybrid_searcher
        self.use_multi_query = use_multi_query
        self.use_rerank = use_rerank
        self.use_compression = use_compression
        self.use_hallucination_check = use_hallucination_check

    def run(self, question: str, k: int = 3) -> dict:
        """Execute the full advanced RAG pipeline."""
        trace: list[str] = []
        trace.append(f"Input question: {question}")

        # Step 1: Query transformation
        if self.use_multi_query:
            queries = [question] + generate_multi_queries(question, n=2)
            trace.append(f"Generated {len(queries)} query variants")
        else:
            queries = [question]

        # Step 2: Hybrid retrieval for each query with RRF fusion
        all_results: dict[str, dict] = {}
        rrf_scores: dict[str, float] = {}
        rrf_constant = 60

        for q in queries:
            results = self.hybrid_searcher.hybrid_search(q, k=k * 2)
            for rank, r in enumerate(results):
                doc_id = r["id"]
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (rrf_constant + rank + 1)
                all_results[doc_id] = r

        # Sort by fused score
        fused_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
        candidates = [all_results[doc_id] for doc_id in fused_ids[: k * 2]]
        trace.append(f"Retrieved {len(candidates)} unique candidates")

        # Step 3: Reranking
        if self.use_rerank:
            candidates = rerank_with_llm(question, candidates, top_k=k)
            trace.append(f"Reranked to top {len(candidates)} documents")
        else:
            candidates = candidates[:k]

        # Step 4: Contextual compression
        if self.use_compression:
            compressed_contexts = []
            for doc in candidates:
                compressed = compress_document(question, doc["content"])
                if compressed:
                    compressed_contexts.append(compressed)
            context = "\n\n".join(compressed_contexts)
            trace.append(f"Compressed {len(candidates)} docs to {len(compressed_contexts)} relevant excerpts")
        else:
            context = "\n\n".join(doc["content"] for doc in candidates)

        # Step 5: Generation
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a helpful technical assistant. Answer the question "
                        "based on the provided context. Cite specific details from "
                        "the context. If the context is insufficient, say so."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Context:\n{context}\n\nQuestion: {question}",
                },
            ],
            temperature=0.1,
        )
        answer = response.choices[0].message.content
        trace.append(f"Generated answer ({len(answer)} chars)")

        # Step 6: Hallucination check
        is_grounded = True
        if self.use_hallucination_check:
            check_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": (
                            f"Is this answer fully grounded in the context? "
                            f"Answer 'yes' or 'no'.\n\n"
                            f"Context: {context[:1500]}\n\n"
                            f"Answer: {answer}"
                        ),
                    }
                ],
                temperature=0.0,
                max_tokens=3,
            )
            is_grounded = "yes" in check_response.choices[0].message.content.lower()
            trace.append(f"Grounded: {is_grounded}")

        return {
            "question": question,
            "answer": answer,
            "is_grounded": is_grounded,
            "sources": [doc["title"] for doc in candidates],
            "trace": trace,
        }


# Build and run the pipeline
pipeline = AdvancedRAGPipeline(
    hybrid_searcher=hybrid_searcher,
    use_multi_query=True,
    use_rerank=True,
    use_compression=True,
    use_hallucination_check=True,
)

# Test with a complex question
result = pipeline.run(
    "What are the best practices for choosing an embedding model and chunking "
    "strategy for a production RAG system?"
)

print("=" * 70)
print("ADVANCED RAG PIPELINE RESULT")
print("=" * 70)
print(f"\nQuestion: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nGrounded: {result['is_grounded']}")
print(f"Sources: {result['sources']}")
print(f"\nPipeline trace:")
for step in result["trace"]:
    print(f"  > {step}")

In [ ]:
# Test another question to see the pipeline adapt
result2 = pipeline.run(
    "How does caching reduce costs in LLM applications and what are the options?"
)

print("=" * 70)
print(f"Question: {result2['question']}")
print(f"\nAnswer:\n{result2['answer']}")
print(f"\nGrounded: {result2['is_grounded']}")
print(f"Sources: {result2['sources']}")
print(f"\nTrace:")
for step in result2["trace"]:
    print(f"  > {step}")

---
## Summary

In this notebook we implemented the key advanced RAG techniques:

| Technique | Purpose | When to Use |
|---|---|---|
| **Query Rewriting** | Optimize query for retrieval | Always -- low cost, high impact |
| **Multi-Query** | Increase recall via diverse phrasings | Diverse user query styles |
| **Query Decomposition** | Handle complex multi-hop questions | Compound questions (compare X and Y) |
| **Hybrid Search** | Combine keyword + semantic matching | Mixed keyword/semantic queries |
| **Reranking** | Improve precision after broad recall | When precision matters |
| **Parent-Child Chunks** | Balance retrieval precision and context | Long documents needing context |
| **HyDE** | Improve retrieval for technical content | When questions differ from document language |
| **Self-Corrective RAG** | Detect and fix retrieval/generation failures | High-stakes answers |
| **Contextual Compression** | Reduce noise in retrieved context | Verbose or long documents |
| **Retrieval Evaluation** | Measure and compare retrieval quality | Always -- instrument from day one |

**Recommendation**: Start with basic RAG + hybrid search + reranking (covers 80% of use cases).
Add parent-child chunks if users complain about incomplete answers. Add query transformation
if retrieval fails on paraphrased queries. Add self-corrective loops only for high-stakes
domains where answer accuracy is critical.

**Next module**: `09-agents.html` -- Building agentic systems with tool use and planning.